# TP 11 — Synthèse : quel algorithme, quand, et pourquoi

Cette fiche rassemble les **contreparties** entre les méthodes du cours et les **situations** où chacune est le bon choix. Chaque ligne renvoie au TP où vous l'avez codée et mesurée. Elle se termine par une liste de pièges propres à la recherche quantitative et par une **auto-évaluation** (réponses dans la version corrigée).

## 1. Carte d'ensemble : de la question à la famille de méthodes

```
Ai-je une variable à prédire (réponse y) ?
├── NON → apprentissage non supervisé
│   ├── trouver des groupes ............................ k-means (TP 2), hiérarchique (TP 3), GMM (TP 4)
│   ├── estimer une loi / un risque / des régimes ...... gaussienne, GMM, histogramme, KDE (TP 4)
│   ├── réduire la dimension / extraire des facteurs ... PCA, quantification vectorielle (TP 10)
│   └── retrouver des voisins / des analogues .......... linear scan, k-d tree (TP 1)
└── OUI → apprentissage supervisé (TP 5 pour le cadre : risque, validation croisée, métriques)
    ├── y continue (régression)
    │   ├── relation ~linéaire, besoin d'interpréter/inférer ...... OLS (TP 6)
    │   ├── beaucoup de prédicteurs, faible rapport signal/bruit .. ridge / lasso (TP 6)
    │   ├── non linéaire, n modéré ............................... kernel ridge (TP 6), k-NN (TP 5)
    │   └── non linéaire, tabulaire, interactions ................ forêt aléatoire (TP 8), MLP (TP 9)
    └── y discrète (classification)
        ├── besoin de probabilités calibrées .................... régression logistique (TP 7)
        ├── frontière non linéaire, n ≲ 10⁵ ...................... SVM à noyau (TP 7)
        ├── tabulaire, variables hétérogènes, bruit .............. forêt aléatoire (TP 8)
        ├── étiquettes propres, apprenants faibles ............... AdaBoost (TP 8)
        └── beaucoup de données structurées (images, signaux) .... CNN (TP 9)
```

**Avant tout modèle** : de bonnes **caractéristiques** (TP 10) et une **référence simple** (moyenne, OLS, règle à une variable) contre laquelle tout gain doit être démontré hors échantillon.

## 2. Contreparties à l'intérieur de chaque chapitre

### TP 1 — Recherche de plus proches voisins
| | linear scan | k-d tree defeatist | k-d tree backtracking |
|---|---|---|---|
| exact | oui | **non** | oui |
| requête | $\Theta(dn)$ | $O(d\log n)$ | $O(2^d\log n)$ → $\Omega(dn)$ |
| choisir si… | $d$ grand, peu de requêtes, toute métrique | aperçu rapide, initialisation | $d$ (intrinsèque) ≲ 10, beaucoup de requêtes |

→ Au-delà : méthodes **approchées** (LSH, HNSW, FAISS). La **dimension intrinsèque** (corrélations) compte plus que le nombre de coordonnées.

### TP 2 — k-means
| | DP 1D | Lloyd + init. aléatoire | Lloyd + k-means++ |
|---|---|---|---|
| optimalité | **globale** | locale, souvent mauvaise | locale, $O(\log k)$-approx. en espérance |
| choisir si… | une seule variable (seuils, régimes) | jamais seul | **défaut**, avec plusieurs redémarrages |

Choix de $k$ : coude (visuel) vs **silhouette** (chiffré, $O(n^2)$) — peuvent diverger si des groupes se chevauchent.

### TP 3 — Clustering hiérarchique
| | single | complete | average | Ward |
|---|---|---|---|---|
| garanties | **stabilité**, bijection avec les ultramétriques | aucune | aucune | même objectif que k-means |
| défaut typique | **effet de chaîne** | coupe les formes allongées, sensible aux outliers | compromis | clusters globulaires de tailles voisines |
| choisir si… | groupes bien séparés de formes quelconques, MST | groupes compacts | défaut générique (corrélations), UPGMA | hiérarchie « k-means » |

### TP 4 — Estimation de densité
| | gaussienne | GMM (EM) | histogramme | KDE |
|---|---|---|---|---|
| vitesse | $\sqrt{d/n}$ | $\sqrt{d/n}$ | $n^{-1/(d+2)}$ | $n^{-2/(d+4)}$ |
| limite | biais si non gaussien (queues !) | maxima locaux, choix de $r$ | discontinu, dimension | dimension, coût, queues extrapolées |
| choisir si… | grande dimension, covariance utile | multimodal, régimes, queues épaisses | exploration rapide 1D-2D | densité lisse en petite dimension |

### TP 5 — Cadre supervisé et validation
| validation | holdout | $r$-fold | LOO | Monte-Carlo | blocs + embargo |
|---|---|---|---|---|---|
| variance de l'estimation | **forte** | faible | faible | faible | — |
| choisir si… | énormes données | **défaut** | petit $n$ ou LOO gratuit (k-NN, ridge) | découpler taille/répétitions | **séries temporelles** |

Métriques : *accuracy* (classes équilibrées) ; précision/rappel/F1 (classe rare) ; ROC/AUC (qualité d'un score, tous seuils) ; log-loss (qualité des probabilités).

### TP 6 — Régression linéaire et au-delà
| | OLS | ridge | lasso | fonctions de base | kernel ridge |
|---|---|---|---|---|---|
| biais / variance | sans biais, variance forte si $d$ grand | biais ↑, variance ↓↓ | biais ↑, sélection | selon la base | selon σ, λ |
| choisir si… | $n\gg d$, inférence (facteurs, $t$-stats) | signaux nombreux, corrélés, effets **denses** | quelques prédicteurs utiles (**sparse**) | non-linéarité de forme connue | non-linéarité lisse, $n\lesssim10^4$ |

Toujours résoudre par **QR** (ou SVD), jamais en inversant $X^\top X$.

### TP 7 — Classification linéaire
| | régression sur indicatrices | logistique | SVM (marge souple) | SVM / logistique à noyau |
|---|---|---|---|---|
| sortie | scores hors $[0,1]$ | **probabilités calibrées** | marge (pas de probabilité) | non linéaire |
| défaut | **masking** (≥ 3 classes) | diverge si séparable → régulariser | $C$ à régler | coût en $n$ |
| choisir si… | jamais | décisions fondées sur des probabilités (trading, crédit), *baseline* | classification pure, grande dimension | frontière non linéaire, $n$ modéré |

### TP 8 — Méthodes d'ensemble
| | arbre | forêt aléatoire (bagging) | AdaBoost (boosting) |
|---|---|---|---|
| réduit | — | la **variance** (moyenne d'arbres profonds décorrélés) | le **biais** (apprenants faibles en série) |
| atouts | interprétable, rapide | robuste, OOB et MDA gratuits, peu de réglages | erreur d'apprentissage → 0 exponentiellement |
| faiblesse | instable | moins interprétable, plancher $\rho\sigma^2$ | **sensible au bruit** (perte exponentielle) |
| choisir si… | expliquer, règles métier | **défaut tabulaire**, données bruitées (finance) | étiquettes propres |

Importance des variables : **MDA** (hors échantillon) plutôt que MDI (biaisée vers les variables continues).

### TP 9 — Réseaux de neurones
| | perceptron | MLP | CNN |
|---|---|---|---|
| capacité | linéaire | approximateur universel | MLP + localité + poids partagés |
| choisir si… | pédagogique | beaucoup de données, interactions complexes | données en grille (images, séries, carnets d'ordres) |

Régularisation : arrêt précoce (toujours), weight decay (= ridge), dropout (grands réseaux). **Toujours vérifier les gradients.**

### TP 10 — Caractéristiques et réduction de dimension
| | PCA | quantification vectorielle | caractéristiques construites |
|---|---|---|---|
| hypothèse | sous-espace **linéaire** | **groupes** | connaissance du domaine |
| choisir si… | facteurs de risque, courbe des taux, débruitage | dictionnaires, ensembles de taille variable | presque toujours : c'est souvent le plus gros gain |

## 3. Comparaison transverse des méthodes supervisées

| méthode | paramétrique | interprétable | probabilités | coût d'apprentissage | coût de prédiction | sensible à l'échelle | grande dimension | données bruitées |
|---|---|---|---|---|---|---|---|---|
| k-NN | non | faible | fréquences | nul | $O(nd)$ | **oui** | mauvais | moyen ($k$ grand) |
| OLS | oui | **forte** | — | $O(nd^2)$ | $O(d)$ | non | mauvais si $d\sim n$ | mauvais |
| ridge / lasso | oui | forte | — | $O(nd^2)$ / itératif | $O(d)$ | **oui** | **bon** | **bon** |
| kernel ridge | non | faible | — | $O(n^3)$ | $O(n)$ | oui | moyen | bon (λ) |
| logistique | oui | forte | **calibrées** | Newton, rapide | $O(d)$ | oui (si régularisée) | bon (régularisée) | bon |
| SVM à noyau | non | faible | non | $O(n^2)$–$O(n^3)$ | $O(\#VS)$ | **oui** | bon | bon ($C$) |
| arbre | non | **forte** | fréquences | $O(dn\log n)$ | $O(\text{prof.})$ | **non** | moyen | mauvais |
| forêt aléatoire | non | moyenne (MDA) | fréquences | $B\times$ arbre | $B\times$ prof. | **non** | bon | **bon** |
| AdaBoost | non | faible | non | $B\times$ souche | $O(B)$ | non | moyen | **mauvais** |
| MLP / CNN | oui (beaucoup) | faible | softmax (souvent mal calibrées) | SGD, coûteux | $O(\#\text{params})$ | **oui** | bon (avec données) | moyen (régularisation) |

## 4. Recherche quantitative : pièges et bonnes pratiques (vus dans le TP)

1. **Fuite d'information temporelle** — standardiser, choisir les hyperparamètres, construire les arbres de recherche **uniquement sur le passé** ; validation croisée **par blocs avec embargo** (TP 1, TP 5).
2. **Faible rapport signal/bruit** — l'OLS sur-apprend ; la régularisation (ridge, feuilles larges, arrêt précoce) est la règle, pas l'exception (TP 6, TP 8, TP 9).
3. **Référence simple** — HAR, règle à une variable, 1/N : un modèle complexe doit les battre hors échantillon (TP 3, TP 5, TP 9).
4. **Queues épaisses et clustering de volatilité** — la gaussienne sous-estime le risque ; une densité iid ignore la dynamique (TP 4).
5. **Probabilités calibrées** — pour dimensionner une position, la calibration *conditionnelle* compte plus que l'*accuracy* (TP 7).
6. **Importance des variables** — MDI biaisée ; préférer MDA / validation (TP 8).
7. **Bruit d'estimation des covariances** — Marchenko-Pastur pour séparer facteurs et bruit ; HRP pour éviter d'inverser une covariance bruitée (TP 3, TP 10).
8. **Standardiser avant toute distance** (k-NN, k-means, SVM, KDE, PCA) (TP 1, TP 2).
9. **Dimension intrinsèque** — les données financières sont corrélées : PCA avant k-NN / kd-tree (TP 1, TP 10).
10. **Vérifier ses implémentations** — tests contre une référence (scikit-learn, force brute), contrôle de gradient (tout le TP).

## 5. Auto-évaluation

Répondez sans regarder vos notebooks, puis comparez avec le corrigé.

**Q1.** Pourquoi la recherche *defeatist* dans un k-d tree peut-elle renvoyer un mauvais voisin, et comment le *backtracking* corrige-t-il ce défaut ?

> ✍️ *Votre réponse :* …

**Q2.** Lloyd converge en un nombre fini d'itérations. Pourquoi ? Converge-t-il vers l'optimum ?

> ✍️ *Votre réponse :* …

**Q3.** Quel est le lien entre single-linkage, arbre couvrant minimal et composantes connexes ?

> ✍️ *Votre réponse :* …

**Q4.** Pourquoi ajuster une gaussienne aux rendements quotidiens conduit-il à sous-estimer la VaR à 99 %, et que proposent GMM et KDE ?

> ✍️ *Votre réponse :* …

**Q5.** Dans la KDE, que se passe-t-il quand σ → 0 et quand σ → ∞ ? Comment choisir σ ?

> ✍️ *Votre réponse :* …

**Q6.** Pourquoi la validation croisée mélangée surestime-t-elle la performance sur une série temporelle ?

> ✍️ *Votre réponse :* …

**Q7.** Ridge ou lasso pour combiner 60 signaux corrélés qui portent tous un peu d'information ? Et si seuls 5 sont utiles ?

> ✍️ *Votre réponse :* …

**Q8.** Pourquoi la régression linéaire sur indicatrices souffre-t-elle du *masking*, et pas la logistique multi-classe ?

> ✍️ *Votre réponse :* …

**Q9.** Que sont les vecteurs de support, et pourquoi le SVM est-il robuste aux points éloignés de la frontière ?

> ✍️ *Votre réponse :* …

**Q10.** Pourquoi une forêt aléatoire tire-t-elle $m<d$ variables à chaque nœud ? Pourquoi ajouter des arbres ne réduit-il pas indéfiniment l'erreur ?

> ✍️ *Votre réponse :* …

**Q11.** Pourquoi AdaBoost est-il déconseillé pour prédire des rendements, alors qu'une forêt fonctionne ?

> ✍️ *Votre réponse :* …

**Q12.** Quelle est la formule de rétropropagation pour une couche dense, et comment vérifier son implémentation ?

> ✍️ *Votre réponse :* …

**Q13.** Qu'apporte le partage de poids d'une convolution par rapport à une couche localement connectée ?

> ✍️ *Votre réponse :* …

**Q14.** Combien de facteurs de risque « réels » dans une matrice de corrélation de 64 actions sur 2600 jours ? Comment le décider ?

> ✍️ *Votre réponse :* …

**Q15.** Pour un problème de classement mensuel d'actions, citez trois choix de méthodologie qui comptent plus que le choix de l'algorithme.

> ✍️ *Votre réponse :* …

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.